# Qwen-27B Matched Fracture Localization and Reciprocal Repair

The preceding expanded-support experiment established a sharp boundary: a reusable weight-change signal can modestly move answers, but the strength of the previously frozen route does not explain which presentation of the same physical transition is solved correctly.

[实验设置与运行说明](../../../docs/experiment_notes/11_handoff_development_and_matching/f1f40f573bcc858f.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_TRACKED_CELLS = 17
_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(total=total, initial=initial, desc=label, unit='cell',
               dynamic_ncols=True, leave=True)
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()
    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )
    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set(); worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close(); _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1); _SETUP_PROGRESS.close()
print(f'Cell progress helper ready: {_TRACKED_CELLS} tracked cells.')


In [ ]:
%%cell_progress 1/17 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install and verify the multimodal environment
import importlib
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Verify GPU and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Lock the matched-fracture protocol
cell_stage('Freezing parent support, pair rules, stages, split, and decisions')
import hashlib
import json
from datetime import datetime, timezone

MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
EXPECTED_LAYER_COUNT = 64
INTERVENTION_BATCH_SIZE = 4
IMAGE_TOKEN_ID_OVERRIDE = None
UPSTREAM_SCOPE = 'options_tokens'

SEED = 20261202
ORIENTATIONS = ('original', 'mirror')
MAPPING_VARIANTS = ('forward', 'reverse')
EDGE_TYPES = ('mapping', 'orientation')
STAGE_SITES = (
    {'stage': 'factor_update', 'layer_index': 48, 'scope': 'options_tokens'},
    {'stage': 'physical_handoff', 'layer_index': 49, 'scope': 'options_tokens'},
    {'stage': 'response_code', 'layer_index': 55, 'scope': 'final_prompt_token'},
    {'stage': 'late_route', 'layer_index': 57, 'scope': 'final_prompt_token'},
    {'stage': 'readout', 'layer_index': 63, 'scope': 'final_prompt_token'},
)
CAPTURE_LAYERS = tuple(site['layer_index'] for site in STAGE_SITES)
STAGE_BY_NAME = {site['stage']: site for site in STAGE_SITES}
MINIMUM_ACTIVE_PAIRS = 8
MINIMUM_CONFIRMATION_FAMILIES_PER_EDGE = 6
BOOTSTRAP_REPLICATES = 10000
ALPHA = 0.05
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.15
NORM_RELATIVE_TOLERANCE = 1e-3

PARENT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_outcomeflip_withincontext_route_confirmation_v2_expanded'
)
PARENT_RUN_PATH = PARENT_ROOT / 'RUN_COMPLETE.json'
PARENT_PROTOCOL_PATH = PARENT_ROOT / 'frozen_protocol.json'
PARENT_MANIFEST_PATH = PARENT_ROOT / 'confirmation_manifest.jsonl'
PARENT_BUNDLE_PATH = PARENT_ROOT / 'confirmation_bundle_manifest.jsonl'
PARENT_RECORDS_PATH = PARENT_ROOT / 'analysis' / 'paired_records.csv'
PARENT_ESTIMANDS_PATH = (
    PARENT_ROOT / 'analysis' / 'outcomeflip_target_estimands.csv'
)

OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_outcomeflip_matched_fracture_reciprocal_v1'
)
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'frozen_protocol.json'
PAIR_MANIFEST_PATH = OUTPUT_ROOT / 'behavior_pair_manifest.jsonl'
ACTIVE_PAIR_MANIFEST_PATH = OUTPUT_ROOT / 'active_pair_manifest.jsonl'
BASELINE_REPLAY_PATH = OUTPUT_ROOT / 'baseline_replay.jsonl'
SCREEN_DIR = OUTPUT_ROOT / 'baseline_screen'
SHARD_DIR = OUTPUT_ROOT / 'intervention_shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
DISCOVERY_SELECTION_PATH = OUTPUT_ROOT / 'discovery_selection.json'
for path in (OUTPUT_ROOT, SCREEN_DIR, SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

def json_default(value):
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, Path): return str(value)
    if torch.is_tensor(value) and value.numel() == 1: return value.item()
    raise TypeError(type(value).__name__)

def write_json_atomic(path, value):
    path = Path(path); temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False,
                   default=json_default) + '\n', encoding='utf-8'
    )
    temporary.replace(path)

def write_jsonl_atomic(path, rows):
    path = Path(path); temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(
                row, sort_keys=True, default=json_default
            ) + '\n')
    temporary.replace(path)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value!r}'.encode('utf-8')).hexdigest()

for required in (
    PARENT_RUN_PATH, PARENT_PROTOCOL_PATH, PARENT_MANIFEST_PATH,
    PARENT_BUNDLE_PATH, PARENT_RECORDS_PATH, PARENT_ESTIMANDS_PATH,
):
    assert required.is_file(), f'Missing prerequisite: {required}'
parent_run = json.loads(PARENT_RUN_PATH.read_text(encoding='utf-8'))
assert parent_run['primary_decision'] == 'outcomeflip_route_gating_not_confirmed'

protocol_identity = {
    'schema_version': 'outcomeflip-matched-fracture-reciprocal-1.0',
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'parent_run_sha256': sha256_file(PARENT_RUN_PATH),
    'parent_manifest_sha256': sha256_file(PARENT_MANIFEST_PATH),
    'parent_bundle_sha256': sha256_file(PARENT_BUNDLE_PATH),
    'parent_records_sha256': sha256_file(PARENT_RECORDS_PATH),
    'pair_rule': (
        'reference solves base and edited endpoints; target solves base but '
        'fails edited endpoint; exactly one presentation factor differs'
    ),
    'anti_copy_rule': 'reference and target edited gold output letters differ',
    'stages': list(STAGE_SITES),
    'split': 'complete family; stable alternating hash; no family leakage',
    'discovery': 'select one stage per edge type by reciprocal score',
    'confirmation': (
        'repair versus own and random; reciprocal induction; '
        'target-letter versus reference-letter control'
    ),
    'cohort_selection_uses_intervention_outputs': False,
    'stage_selection_uses_discovery_interventions': True,
    'confirmation_results_used_for_stage_selection': False,
    'bootstrap_unit': 'complete physical family',
    'seed': SEED,
}
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous['identity'] == protocol_identity, (
        'Existing protocol differs; use a new output version.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
print('Parent decision:', parent_run['primary_decision'])
print('Frozen stages:', [(s['stage'], s['layer_index'], s['scope']) for s in STAGE_SITES])
print('Output:', OUTPUT_ROOT)


In [ ]:
%%cell_progress 5/17 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


In [ ]:
%%cell_progress 6/17 Build behavior-matched discordant pairs
cell_stage('Pairing successful and failed presentations without causal outcomes')
from collections import Counter, defaultdict
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

factor_manifest = read_jsonl(PARENT_MANIFEST_PATH)
row_by_family = {str(row['family_id']): row for row in factor_manifest}
parent_records = pd.read_csv(PARENT_RECORDS_PATH)
parent_estimands = pd.read_csv(PARENT_ESTIMANDS_PATH)

def label_for(semantic, mapping_variant):
    order = (
        ('left', 'balanced', 'right')
        if mapping_variant == 'forward'
        else ('right', 'balanced', 'left')
    )
    return ('A', 'B', 'C')[order.index(str(semantic))]

baseline_conditions = parent_records[
    parent_records['condition'].isin(['clean', 'natural_hybrid'])
].copy()
behavior_cells = []
group_columns = [
    'family_id', 'target_key', 'orientation', 'mapping_variant'
]
for keys, current in baseline_conditions.groupby(group_columns, sort=True):
    by_condition = {
        str(row['condition']): row
        for row in current.to_dict(orient='records')
    }
    if set(by_condition) != {'clean', 'natural_hybrid'}:
        continue
    clean = by_condition['clean']; natural = by_condition['natural_hybrid']
    behavior_cells.append({
        **dict(zip(group_columns, keys)),
        'bundle_id': f'{keys[0]}::{keys[1]}',
        'clean_gold': str(clean['gold_semantic']),
        'natural_gold': str(natural['gold_semantic']),
        'clean_prediction': str(clean['predicted_semantic']),
        'natural_prediction': str(natural['predicted_semantic']),
        'clean_correct': bool(clean['predicted_semantic'] == clean['gold_semantic']),
        'natural_correct': bool(
            natural['predicted_semantic'] == natural['gold_semantic']
        ),
    })
behavior_frame = pd.DataFrame(behavior_cells)
assert len(behavior_frame) == 4 * len(read_jsonl(PARENT_BUNDLE_PATH))

pair_candidates = []
for bundle_id, current in behavior_frame.groupby('bundle_id', sort=True):
    records = current.to_dict(orient='records')
    references = [
        row for row in records if row['clean_correct'] and row['natural_correct']
    ]
    targets = [
        row for row in records if row['clean_correct'] and not row['natural_correct']
    ]
    for target in targets:
        for reference in references:
            orientation_differs = (
                reference['orientation'] != target['orientation']
            )
            mapping_differs = (
                reference['mapping_variant'] != target['mapping_variant']
            )
            if int(orientation_differs) + int(mapping_differs) != 1:
                continue
            edge_type = 'orientation' if orientation_differs else 'mapping'
            reference_label = label_for(
                reference['natural_gold'], reference['mapping_variant']
            )
            target_label = label_for(
                target['natural_gold'], target['mapping_variant']
            )
            if reference_label == target_label:
                continue
            pair_candidates.append({
                'family_id': str(reference['family_id']),
                'target_key': str(reference['target_key']),
                'bundle_id': bundle_id,
                'edge_type': edge_type,
                'reference_orientation': str(reference['orientation']),
                'reference_mapping': str(reference['mapping_variant']),
                'target_orientation': str(target['orientation']),
                'target_mapping': str(target['mapping_variant']),
                'reference_base_gold': str(reference['clean_gold']),
                'reference_edited_gold': str(reference['natural_gold']),
                'target_base_gold': str(target['clean_gold']),
                'target_edited_gold': str(target['natural_gold']),
                'reference_edited_label': reference_label,
                'target_edited_label': target_label,
                'target_error_prediction': str(target['natural_prediction']),
                'target_error_label': label_for(
                    target['natural_prediction'], target['mapping_variant']
                ),
                'stale_error': bool(
                    target['natural_prediction'] == target['clean_gold']
                ),
            })

selected_pairs = []
grouped_candidates = defaultdict(list)
for row in pair_candidates:
    grouped_candidates[(row['bundle_id'], row['edge_type'])].append(row)
for key, candidates in sorted(grouped_candidates.items()):
    candidates.sort(key=lambda row: (
        not row['stale_error'], stable_key(('pair', key, row))
    ))
    selected_pairs.append(candidates[0])

family_ids = sorted(
    {row['family_id'] for row in selected_pairs}, key=stable_key
)
family_split = {
    family_id: ('discovery' if index % 2 == 0 else 'confirmation')
    for index, family_id in enumerate(family_ids)
}
for row in selected_pairs:
    row['split'] = family_split[row['family_id']]
    row['pair_id'] = 'fracture_' + stable_key((
        row['bundle_id'], row['edge_type'],
        row['reference_orientation'], row['reference_mapping'],
        row['target_orientation'], row['target_mapping'],
    ))[:20]
selected_pairs.sort(key=lambda row: row['pair_id'])

assert selected_pairs, 'No single-factor update-error pairs were available.'
assert all(
    row['reference_edited_label'] != row['target_edited_label']
    for row in selected_pairs
)
assert not (
    {row['family_id'] for row in selected_pairs if row['split'] == 'discovery'}
    & {row['family_id'] for row in selected_pairs if row['split'] == 'confirmation'}
)
write_jsonl_atomic(PAIR_MANIFEST_PATH, selected_pairs)
pair_manifest = selected_pairs

loader_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR, output_dir=OUTPUT_ROOT / '_unused_loader',
    model_id=MODEL_ID, model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,), question_kinds=('outcome',),
    splits=('probe_train', 'probe_dev', 'probe_test'), batch_size=1,
    generation_every=0, seed=SEED,
)
outcome_examples = tuple(load_behavior_examples(loader_config))
outcome_by_scene = {}
for example in outcome_examples:
    outcome_by_scene.setdefault(str(example.scene_id), example)
required_scene_ids = {
    str(role['scene_id'])
    for row in factor_manifest for roles in row['orientation_roles'].values()
    for role in roles.values()
}
assert not (required_scene_ids - set(outcome_by_scene))

print('Behavior-defined pair candidates / selected:',
      len(pair_candidates), len(pair_manifest))
print('Families:', len(family_ids))
display(pd.DataFrame(pair_manifest).groupby(
    ['split', 'edge_type', 'stale_error']
).size().rename('pairs'))


In [ ]:
%%cell_progress 7/17 Load pinned Qwen-27B and capture stages
cell_stage('Loading Qwen-27B and resolving the five frozen layers')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, module_path):
    value = root
    for component in module_path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers', 'model.layers',
    )
    for module_path in candidates:
        try:
            layers = resolve_module_path(model, module_path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return module_path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
CAPTURE_MODULES = {
    layer_index: DECODER_LAYERS[layer_index]
    for layer_index in CAPTURE_LAYERS
}
print('Layer path:', LAYER_PATH)
print('Capture layers:', CAPTURE_LAYERS)


In [ ]:
%%cell_progress 8/17 Define paired prompts, labels, and exact masks
cell_stage('Defining answer mappings and label-aware scoring')
import math
from dataclasses import replace as dataclass_replace

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = 'Which way will the balance scale tip?'
SEMANTIC_TEXT = {
    'left': 'left side', 'balanced': 'equal', 'right': 'right side',
}

def mapping_values(mapping_variant):
    semantics = (
        ('left', 'balanced', 'right')
        if mapping_variant == 'forward'
        else ('right', 'balanced', 'left')
    )
    return tuple(zip(('A', 'B', 'C'), semantics))

def custom_prompt(mapping):
    options = '\n'.join(
        f'{label}. {SEMANTIC_TEXT[semantic]}'
        for label, semantic in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT,
        'Question: ' + QUESTION_TEXT,
        'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option '
        'letter and nothing else. Valid replies are: A, B, C.',
    ])

def role_record(row, orientation, role_name):
    return row['orientation_roles'][orientation][role_name]

def changed_side_for(row, orientation):
    return row['orientation_changed_side'][orientation]

def make_example(row, orientation, mapping_variant, role_name):
    mapping = mapping_values(mapping_variant)
    role = role_record(row, orientation, role_name)
    scene_id = str(role['scene_id'])
    semantics = tuple(value for _, value in mapping)
    labels = tuple(label for label, _ in mapping)
    outcome = str(role['outcome'])
    base = outcome_by_scene[scene_id]
    return dataclass_replace(
        base,
        example_id=(
            f"weight_factorial:{row['family_id']}:{orientation}:"
            f"{mapping_variant}:{role_name}"
        ),
        question_id=(
            f"weight_factorial_{row['family_id']}_{orientation}_{mapping_variant}"
        ),
        question_kind='outcome', variant='weight_mapping_side_factorial',
        prompt=custom_prompt(mapping), labels=labels,
        option_text=tuple(SEMANTIC_TEXT[value] for value in semantics),
        option_semantics=semantics,
        gold_label=labels[semantics.index(outcome)], gold_semantic=outcome,
    )

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state'):
        hidden = output.last_hidden_state
    else:
        raise TypeError(type(output).__name__)
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples],
            add_generation_prompt=True,
        )),
        fixed_prefix,
    )
    return {'examples': examples, 'inputs': inputs, 'token_by_label': token_by_label}

def discover_masks(prepared):
    assert len(prepared['examples']) == 1
    example = prepared['examples'][0]
    ids = prepared['inputs']['input_ids'][0].detach().cpu()
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    tokenizer = getattr(adapter.processor, 'tokenizer', adapter.processor)
    tokens = tokenizer.convert_ids_to_tokens(ids.tolist())
    image_ids = set()
    if IMAGE_TOKEN_ID_OVERRIDE is not None:
        values = IMAGE_TOKEN_ID_OVERRIDE if isinstance(
            IMAGE_TOKEN_ID_OVERRIDE, (list, tuple)
        ) else [IMAGE_TOKEN_ID_OVERRIDE]
        image_ids.update(int(value) for value in values)
    for token_id, token_text in zip(ids.tolist(), tokens):
        lower = str(token_text).lower()
        if any(marker in lower for marker in ('image', 'vision', '<img', 'img>')):
            image_ids.add(int(token_id))
    image = torch.tensor(
        [int(token_id) in image_ids for token_id in ids.tolist()], dtype=torch.bool
    ) & attention
    assert bool(image.any()), 'No image tokens detected.'
    final = torch.zeros_like(attention)
    final[int(torch.where(attention)[0][-1])] = True
    all_text = attention & ~image & ~final

    encoding = tokenizer(
        example.prompt, add_special_tokens=False, return_offsets_mapping=True,
    )
    prompt_ids = list(encoding['input_ids'])
    offsets = list(encoding['offset_mapping'])
    id_list = ids.tolist()
    starts = [
        start for start in range(len(ids) - len(prompt_ids) + 1)
        if id_list[start:start + len(prompt_ids)] == prompt_ids
    ]
    assert len(starts) == 1
    prompt_start = starts[0]
    sections = example.prompt.split('\n\n')
    names = (
        'intro_tokens', 'rule_tokens', 'question_tokens',
        'options_tokens', 'answer_instruction_tokens',
    )
    assert len(sections) == len(names)
    masks = {
        'valid_tokens': attention, 'image_tokens': image,
        'all_text_except_final': all_text, 'final_prompt_token': final,
    }
    spans = []
    cursor = 0
    for name, section in zip(names, sections):
        start = example.prompt.find(section, cursor)
        end = start + len(section)
        cursor = end
        spans.append((name, start, end))
        masks[name] = torch.zeros_like(attention)
    for offset_index, (token_start, token_end) in enumerate(offsets):
        overlap, name = max(
            (max(0, min(token_end, end) - max(token_start, start)), name)
            for name, start, end in spans
        )
        full_index = prompt_start + offset_index
        if overlap > 0 and bool(all_text[full_index]):
            masks[name][full_index] = True
    covered = torch.zeros_like(attention)
    for name in names:
        covered |= masks[name]
    masks['separator_chat_tokens'] = all_text & ~covered
    assert torch.equal(
        masks['options_tokens'] | masks['intro_tokens'] | masks['rule_tokens']
        | masks['question_tokens'] | masks['answer_instruction_tokens']
        | masks['separator_chat_tokens'], all_text,
    )
    masks['tokens'] = tokens
    return masks

def assert_aligned(prepared_by_name):
    names = list(prepared_by_name)
    reference = prepared_by_name[names[0]]
    reference_masks = discover_masks(reference)
    for name in names[1:]:
        current = prepared_by_name[name]
        for key in ('input_ids', 'attention_mask'):
            assert torch.equal(reference['inputs'][key], current['inputs'][key])
        masks = discover_masks(current)
        for scope in (UPSTREAM_SCOPE, 'final_prompt_token'):
            assert torch.equal(reference_masks[scope], masks[scope])
    return reference_masks

def score_records(prepared, logits):
    records = []
    for batch_index, example in enumerate(prepared['examples']):
        logprobs = torch.log_softmax(logits[batch_index, -1, :].float(), dim=-1)
        scores = [
            float(logprobs[prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        predicted_index = int(np.argmax(scores))
        records.append({
            'scene_id': str(example.scene_id),
            'example_id': str(example.example_id),
            'candidate_semantics': list(example.option_semantics),
            'candidate_labels': list(example.labels),
            'candidate_logprobs': scores,
            'candidate_probabilities': [value / sum(weights) for value in weights],
            'predicted_semantic': example.option_semantics[predicted_index],
            'predicted_label': example.labels[predicted_index],
            'gold_semantic': example.gold_semantic,
            'gold_label': example.gold_label,
        })
    return records

def centered_error(first, second):
    first = np.asarray(first, dtype=float)
    second = np.asarray(second, dtype=float)
    return float(np.max(np.abs(
        (first - first.mean()) - (second - second.mean())
    )))

def forward_score_prepared(prepared):
    with torch.inference_mode():
        output = adapter.model(**prepared['inputs'], use_cache=False)
    return score_records(prepared, output.logits)

print('Fully paired prompts and masks ready.')


In [ ]:
%%cell_progress 9/17 Define stage capture and reciprocal interventions
cell_stage('Defining layerwise update transport with matched controls')

def forward_capture(prepared):
    assert len(prepared['examples']) == 1
    captured = {}
    handles = []
    for layer_index, module in CAPTURE_MODULES.items():
        def capture(_module, _inputs, output, layer_index=layer_index):
            captured[layer_index] = (
                output_hidden_tensor(output)[0].detach().cpu().to(torch.bfloat16)
            )
        handles.append(module.register_forward_hook(capture))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles: handle.remove()
    assert set(captured) == set(CAPTURE_LAYERS)
    return score_records(prepared, output.logits)[0], captured

def forward_score(example):
    prepared = prepare_examples([example])
    return forward_score_prepared(prepared)[0]

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def centered_error(first, second):
    first = np.asarray(first, dtype=float); second = np.asarray(second, dtype=float)
    return float(np.max(np.abs(
        (first - first.mean()) - (second - second.mean())
    )))

def masked_delta(changed, base, mask):
    result = torch.zeros_like(changed, dtype=torch.bfloat16)
    selected = mask.bool()
    result[selected] = (
        changed[selected].float() - base[selected].float()
    ).to(torch.bfloat16)
    return result

def selected_norm(tensor, mask):
    return float(torch.linalg.vector_norm(tensor[mask.bool()].float()))

def match_norm(tensor, reference, mask):
    result = torch.zeros_like(tensor, dtype=torch.float32)
    selected = mask.bool(); values = tensor[selected].float()
    current = torch.linalg.vector_norm(values)
    target = torch.linalg.vector_norm(reference[selected].float())
    if float(current) > 0 and float(target) > 0:
        values = values * (target / current)
    else:
        values.zero_()
    result[selected] = values
    return result.to(torch.bfloat16)

def random_like(reference, mask, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    result = torch.zeros_like(reference, dtype=torch.float32)
    selected = mask.bool()
    values = torch.randn(
        reference[selected].shape, generator=generator, dtype=torch.float32
    )
    target = torch.linalg.vector_norm(reference[selected].float())
    values = values * (target / torch.linalg.vector_norm(values))
    result[selected] = values
    return result.to(torch.bfloat16)

def build_pair_examples(pair):
    row = row_by_family[pair['family_id']]
    base_role = f"target_{pair['target_key']}_base"
    edited_role = f"target_{pair['target_key']}_hybrid"
    return {
        'reference_base': make_example(
            row, pair['reference_orientation'], pair['reference_mapping'], base_role
        ),
        'reference_edited': make_example(
            row, pair['reference_orientation'], pair['reference_mapping'], edited_role
        ),
        'target_base': make_example(
            row, pair['target_orientation'], pair['target_mapping'], base_role
        ),
        'target_edited': make_example(
            row, pair['target_orientation'], pair['target_mapping'], edited_role
        ),
    }

def prepare_pair(pair):
    examples = build_pair_examples(pair)
    prepared = {name: prepare_examples([example]) for name, example in examples.items()}
    masks = {name: discover_masks(value) for name, value in prepared.items()}
    shapes = {tuple(value['inputs']['input_ids'].shape) for value in prepared.values()}
    assert len(shapes) == 1
    for site in STAGE_SITES:
        scope = site['scope']
        reference = masks['target_base'][scope]
        assert all(torch.equal(reference, current[scope]) for current in masks.values())
        assert bool(reference.any())
    return examples, prepared, masks

def score_stage_conditions(example, stage, conditions, mask_override=None):
    results = []
    layer_index = int(stage['layer_index'])
    for start in range(0, len(conditions), INTERVENTION_BATCH_SIZE):
        actual = list(conditions[start:start + INTERVENTION_BATCH_SIZE])
        padded = actual + [actual[-1]] * (INTERVENTION_BATCH_SIZE - len(actual))
        prepared = prepare_examples([example] * INTERVENTION_BATCH_SIZE)
        single = prepare_examples([example])
        masks = discover_masks(single)
        selected_cpu = (
            mask_override.bool() if mask_override is not None
            else masks[stage['scope']].bool()
        )
        def hook(_module, _inputs, output):
            hidden = output_hidden_tensor(output)
            selected = selected_cpu.to(hidden.device)
            modified = hidden.clone()
            for batch_index, condition in enumerate(padded):
                payload = condition['payload'].to(hidden.device, hidden.dtype)
                if condition['mode'] == 'add':
                    modified[batch_index, selected] = (
                        hidden[batch_index, selected] + payload[selected]
                    )
                elif condition['mode'] == 'replace':
                    modified[batch_index, selected] = payload[selected]
                else:
                    raise ValueError(condition['mode'])
            return replace_output_hidden(output, modified)
        handle = DECODER_LAYERS[layer_index].register_forward_hook(hook)
        try:
            with torch.inference_mode():
                output = adapter.model(**prepared['inputs'], use_cache=False)
        finally:
            handle.remove()
        current = score_records(prepared, output.logits)[:len(actual)]
        for record, condition in zip(current, actual):
            record['condition'] = condition['condition']
            record['payload_norm'] = selected_norm(
                condition['payload'], selected_cpu
            )
            results.append(record)
    return results

def counterfactual_progress(record, base, edited):
    current = centered_scores(record); start = centered_scores(base)
    destination = centered_scores(edited); direction = destination - start
    denominator = float(direction @ direction)
    assert denominator > 1e-12
    return float((current - start) @ direction / denominator)

def label_margin(record, positive_label, negative_label):
    labels = list(record['candidate_labels'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(values[labels.index(positive_label)] - values[labels.index(negative_label)])

print('Stage capture and reciprocal intervention helpers ready.')


In [ ]:
%%cell_progress 10/17 Replay and freeze the active discordant cohort
cell_stage('Replaying reference and target endpoints with resumable screens')

def screen_key(pair, role_name):
    return stable_key(('baseline', pair['pair_id'], role_name))[:24]

screen_requests = {}
for pair in pair_manifest:
    for role_name, example in build_pair_examples(pair).items():
        screen_requests[screen_key(pair, role_name)] = (pair, role_name, example)

for key, (pair, role_name, example) in tqdm(
    screen_requests.items(), desc='Matched endpoint replay', unit='endpoint'
):
    path = SCREEN_DIR / f'{key}.json'
    if path.is_file(): continue
    record = forward_score(example)
    record.update({'pair_id': pair['pair_id'], 'role_name': role_name})
    write_json_atomic(path, record)

screen_cache = {
    path.stem: json.loads(path.read_text(encoding='utf-8'))
    for path in SCREEN_DIR.glob('*.json')
}
replay_rows = []
active_pairs = []
for pair in pair_manifest:
    records = {
        role_name: screen_cache[screen_key(pair, role_name)]
        for role_name in (
            'reference_base', 'reference_edited',
            'target_base', 'target_edited',
        )
    }
    replay_ok = bool(
        records['reference_base']['predicted_semantic']
        == records['reference_base']['gold_semantic']
        and records['reference_edited']['predicted_semantic']
        == records['reference_edited']['gold_semantic']
        and records['target_base']['predicted_semantic']
        == records['target_base']['gold_semantic']
        and records['target_edited']['predicted_semantic']
        != records['target_edited']['gold_semantic']
        and records['reference_edited']['gold_label']
        != records['target_edited']['gold_label']
    )
    replay_rows.append({
        'pair_id': pair['pair_id'], 'family_id': pair['family_id'],
        'edge_type': pair['edge_type'], 'split': pair['split'],
        'replay_ok': replay_ok,
        'reference_edited_prediction': records['reference_edited']['predicted_semantic'],
        'target_edited_prediction': records['target_edited']['predicted_semantic'],
        'target_edited_gold': records['target_edited']['gold_semantic'],
    })
    if replay_ok: active_pairs.append(pair)

replay_frame = pd.DataFrame(replay_rows)
write_jsonl_atomic(BASELINE_REPLAY_PATH, replay_rows)
write_jsonl_atomic(ACTIVE_PAIR_MANIFEST_PATH, active_pairs)
active_pair_manifest = active_pairs
REPLAY_CONCORDANCE = float(replay_frame['replay_ok'].mean())
assert active_pair_manifest, 'No discordant pair replayed its frozen behavior.'
ACTIVE_SUPPORT_SHORTFALL = bool(
    len(active_pair_manifest) < MINIMUM_ACTIVE_PAIRS
)
print('Candidate / active pairs:', len(pair_manifest), len(active_pair_manifest))
print('Replay concordance:', REPLAY_CONCORDANCE)
if ACTIVE_SUPPORT_SHORTFALL:
    print(
        'Support note: only', len(active_pair_manifest),
        'active pairs replayed versus the planned minimum of',
        MINIMUM_ACTIVE_PAIRS,
        '; the notebook will continue but confirmation gates remain closed.'
    )
display(pd.DataFrame(active_pair_manifest).groupby(
    ['split', 'edge_type', 'stale_error']
).size().rename('pairs'))


In [ ]:
%%cell_progress 11/17 Run alignment, reconstruction, and anti-copy smoke tests
cell_stage('Checking full-state reconstruction, masks, norms, and labels')

smoke_pair = active_pair_manifest[0]
smoke_examples, smoke_prepared, smoke_masks = prepare_pair(smoke_pair)
smoke_scores = {}; smoke_hidden = {}
for name, prepared in smoke_prepared.items():
    smoke_scores[name], smoke_hidden[name] = forward_capture(prepared)

assert (
    smoke_scores['reference_edited']['gold_label']
    != smoke_scores['target_edited']['gold_label']
)
smoke_stage = STAGE_SITES[0]
layer_index = smoke_stage['layer_index']
valid_mask = smoke_masks['target_base']['valid_tokens']
reconstruction = score_stage_conditions(
    smoke_examples['target_base'], smoke_stage, [{
        'condition': 'full_natural_reconstruction',
        'payload': smoke_hidden['target_edited'][layer_index],
        'mode': 'replace',
    }], mask_override=valid_mask,
)[0]
reconstruction_error = centered_error(
    reconstruction['candidate_logprobs'],
    smoke_scores['target_edited']['candidate_logprobs'],
)
assert reconstruction_error < SMOKE_CENTERED_LOGPROB_TOLERANCE, reconstruction_error

scope_mask = smoke_masks['target_base'][smoke_stage['scope']]
target_delta = masked_delta(
    smoke_hidden['target_edited'][layer_index],
    smoke_hidden['target_base'][layer_index], scope_mask,
)
random_delta = random_like(target_delta, scope_mask, ('smoke', smoke_pair['pair_id']))
relative_norm_error = abs(
    selected_norm(random_delta, scope_mask) - selected_norm(target_delta, scope_mask)
) / max(selected_norm(target_delta, scope_mask), 1e-12)
assert relative_norm_error < NORM_RELATIVE_TOLERANCE
print('Smoke pair:', smoke_pair['pair_id'], smoke_pair['edge_type'])
print('Full-state reconstruction error:', reconstruction_error)
print('Matched random norm error:', relative_norm_error)


## Stagewise reciprocal transport

The target base is a correctly solved starting point.


In [ ]:
%%cell_progress 12/17 Run stagewise reciprocal repair and induction
cell_stage('Processing active pairs with resumable pair shards')

capture_identity = {
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'active_pair_sha256': sha256_file(ACTIVE_PAIR_MANIFEST_PATH),
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'stages': list(STAGE_SITES),
}
CAPTURE_SHA256 = 'sha256:' + hashlib.sha256(
    json.dumps(capture_identity, sort_keys=True).encode('utf-8')
).hexdigest()

def decorate(record, pair, stage, recipient, base, edited):
    record.update({
        'pair_id': pair['pair_id'], 'family_id': pair['family_id'],
        'bundle_id': pair['bundle_id'], 'edge_type': pair['edge_type'],
        'split': pair['split'], 'stale_error': pair['stale_error'],
        'stage': stage['stage'], 'layer_index': stage['layer_index'],
        'scope': stage['scope'], 'recipient': recipient,
        'counterfactual_progress': counterfactual_progress(record, base, edited),
    })
    return record

for pair in tqdm(
    active_pair_manifest, desc='Matched fracture pairs', unit='pair'
):
    shard_path = SHARD_DIR / f"{pair['pair_id']}.json"
    if shard_path.is_file():
        existing = json.loads(shard_path.read_text(encoding='utf-8'))
        assert existing['capture_identity_sha256'] == CAPTURE_SHA256
        continue
    examples, prepared, masks = prepare_pair(pair)
    scores = {}; hidden = {}
    for name, current in prepared.items():
        scores[name], hidden[name] = forward_capture(current)
    assert scores['reference_base']['predicted_label'] == scores['reference_base']['gold_label']
    assert scores['reference_edited']['predicted_label'] == scores['reference_edited']['gold_label']
    assert scores['target_base']['predicted_label'] == scores['target_base']['gold_label']
    assert scores['target_edited']['predicted_label'] != scores['target_edited']['gold_label']

    pair_records = []
    for stage in STAGE_SITES:
        layer_index = int(stage['layer_index']); scope = stage['scope']
        target_mask = masks['target_base'][scope]
        reference_mask = masks['reference_base'][scope]
        assert torch.equal(target_mask, reference_mask)
        reference_delta = masked_delta(
            hidden['reference_edited'][layer_index],
            hidden['reference_base'][layer_index], reference_mask,
        )
        target_delta = masked_delta(
            hidden['target_edited'][layer_index],
            hidden['target_base'][layer_index], target_mask,
        )
        reference_for_target = match_norm(reference_delta, target_delta, target_mask)
        target_for_reference = match_norm(target_delta, reference_delta, reference_mask)
        target_random = random_like(
            target_delta, target_mask,
            ('target-random', pair['pair_id'], stage['stage']),
        )
        reference_random = random_like(
            reference_delta, reference_mask,
            ('reference-random', pair['pair_id'], stage['stage']),
        )

        target_conditions = [
            {'condition': 'target_own_update', 'payload': target_delta, 'mode': 'add'},
            {'condition': 'reference_update', 'payload': reference_for_target, 'mode': 'add'},
            {'condition': 'reference_raw_update', 'payload': reference_delta, 'mode': 'add'},
            {'condition': 'random_update', 'payload': target_random, 'mode': 'add'},
            {'condition': 'reverse_reference_update', 'payload': -reference_for_target, 'mode': 'add'},
        ]
        target_records = score_stage_conditions(
            examples['target_base'], stage, target_conditions
        )
        for record in target_records:
            decorate(
                record, pair, stage, 'target',
                scores['target_base'], scores['target_edited'],
            )
            record['desired_label'] = scores['target_edited']['gold_label']
            record['reference_label'] = scores['reference_edited']['gold_label']
            record['desired_vs_reference_margin'] = label_margin(
                record, record['desired_label'], record['reference_label']
            )
            record['reaches_desired_label'] = bool(
                record['predicted_label'] == record['desired_label']
            )
            record['copies_reference_label'] = bool(
                record['predicted_label'] == record['reference_label']
            )
            pair_records.append(record)

        reference_conditions = [
            {'condition': 'reference_own_update', 'payload': reference_delta, 'mode': 'add'},
            {'condition': 'target_failed_update', 'payload': target_for_reference, 'mode': 'add'},
            {'condition': 'random_failed_update', 'payload': reference_random, 'mode': 'add'},
        ]
        reference_records = score_stage_conditions(
            examples['reference_base'], stage, reference_conditions
        )
        for record in reference_records:
            decorate(
                record, pair, stage, 'reference',
                scores['reference_base'], scores['reference_edited'],
            )
            record['desired_label'] = scores['reference_edited']['gold_label']
            record['reference_label'] = scores['target_edited']['gold_label']
            record['desired_vs_reference_margin'] = label_margin(
                record, record['desired_label'], record['reference_label']
            )
            record['reaches_desired_label'] = bool(
                record['predicted_label'] == record['desired_label']
            )
            record['copies_reference_label'] = bool(
                record['predicted_label'] == record['reference_label']
            )
            pair_records.append(record)

    write_json_atomic(shard_path, {
        'schema_version': 'outcomeflip-fracture-pair-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'pair': pair, 'records': pair_records,
    })
    del hidden
    torch.cuda.empty_cache()

assert len(list(SHARD_DIR.glob('*.json'))) == len(active_pair_manifest)
write_json_atomic(OUTPUT_ROOT / 'capture_complete.json', {
    'capture_identity': capture_identity,
    'capture_identity_sha256': CAPTURE_SHA256,
    'pairs': len(active_pair_manifest),
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
})
print('Complete pair shards:', len(active_pair_manifest))


In [ ]:
%%cell_progress 13/17 Assemble pair-stage fracture estimands
cell_stage('Constructing repair, induction, specificity, and anti-copy contrasts')
import gc
try:
    del adapter; del DECODER_LAYERS
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

records = []
for path in sorted(SHARD_DIR.glob('*.json')):
    document = json.loads(path.read_text(encoding='utf-8'))
    assert document['capture_identity_sha256'] == CAPTURE_SHA256
    records.extend(document['records'])
record_frame = pd.DataFrame(records)
record_frame.to_csv(ANALYSIS_DIR / 'intervention_records.csv', index=False)

estimand_rows = []
for (pair_id, stage_name), current in record_frame.groupby(
    ['pair_id', 'stage'], sort=True
):
    by_condition = {
        str(row['condition']): row for row in current.to_dict(orient='records')
    }
    expected = {
        'target_own_update', 'reference_update', 'reference_raw_update',
        'random_update', 'reverse_reference_update',
        'reference_own_update', 'target_failed_update',
        'random_failed_update',
    }
    assert set(by_condition) == expected
    target_own = by_condition['target_own_update']
    repair = by_condition['reference_update']
    raw_repair = by_condition['reference_raw_update']
    target_random = by_condition['random_update']
    reverse = by_condition['reverse_reference_update']
    reference_own = by_condition['reference_own_update']
    failed = by_condition['target_failed_update']
    reference_random = by_condition['random_failed_update']
    estimand_rows.append({
        'pair_id': pair_id, 'family_id': repair['family_id'],
        'bundle_id': repair['bundle_id'], 'edge_type': repair['edge_type'],
        'split': repair['split'], 'stale_error': bool(repair['stale_error']),
        'stage': stage_name, 'layer_index': int(repair['layer_index']),
        'scope': repair['scope'],
        'repair_vs_own': float(
            repair['counterfactual_progress'] - target_own['counterfactual_progress']
        ),
        'repair_vs_random': float(
            repair['counterfactual_progress'] - target_random['counterfactual_progress']
        ),
        'repair_vs_reverse': float(
            repair['counterfactual_progress'] - reverse['counterfactual_progress']
        ),
        'raw_repair_vs_random': float(
            raw_repair['counterfactual_progress'] - target_random['counterfactual_progress']
        ),
        'induction_success_minus_failed': float(
            reference_own['counterfactual_progress'] - failed['counterfactual_progress']
        ),
        'failed_vs_random': float(
            failed['counterfactual_progress'] - reference_random['counterfactual_progress']
        ),
        'reciprocal_score': float(0.5 * (
            repair['counterfactual_progress'] - target_own['counterfactual_progress']
            + reference_own['counterfactual_progress']
            - failed['counterfactual_progress']
        )),
        'target_alignment_vs_own': float(
            repair['desired_vs_reference_margin']
            - target_own['desired_vs_reference_margin']
        ),
        'target_alignment_vs_random': float(
            repair['desired_vs_reference_margin']
            - target_random['desired_vs_reference_margin']
        ),
        'repair_success_advantage': float(
            repair['reaches_desired_label'] - target_own['reaches_desired_label']
        ),
        'repair_minus_copy': float(
            repair['reaches_desired_label'] - repair['copies_reference_label']
        ),
        'induction_success_advantage': float(
            reference_own['reaches_desired_label']
            - failed['reaches_desired_label']
        ),
        'target_update_norm': float(target_own['payload_norm']),
        'reference_update_norm': float(repair['payload_norm']),
        'reference_raw_norm': float(raw_repair['payload_norm']),
    })

estimand_frame = pd.DataFrame(estimand_rows)
assert len(estimand_frame) == len(active_pair_manifest) * len(STAGE_SITES)
assert estimand_frame.groupby('pair_id').size().eq(len(STAGE_SITES)).all()
estimand_frame.to_csv(ANALYSIS_DIR / 'pair_stage_estimands.csv', index=False)
print('Pairs / stages / estimands:',
      estimand_frame['pair_id'].nunique(), len(STAGE_SITES), len(estimand_frame))


In [ ]:
%%cell_progress 14/17 Select one stage per edge type on discovery families
cell_stage('Localizing the strongest reciprocal stage without confirmation families')

DISCOVERY_METRICS = (
    'repair_vs_own', 'repair_vs_random',
    'induction_success_minus_failed', 'target_alignment_vs_own',
)
discovery_frame = estimand_frame[estimand_frame['split'] == 'discovery'].copy()
confirmation_frame = estimand_frame[estimand_frame['split'] == 'confirmation'].copy()
assert not (
    set(discovery_frame['family_id']) & set(confirmation_frame['family_id'])
)

selection_rows = []
frozen_stage_by_edge = {}
for edge_type in EDGE_TYPES:
    current = discovery_frame[discovery_frame['edge_type'] == edge_type]
    if current.empty:
        continue
    summaries = current.groupby(
        ['stage', 'layer_index', 'scope'], as_index=False
    )[list(DISCOVERY_METRICS) + ['reciprocal_score']].mean()
    summaries['selection_score'] = (
        summaries['repair_vs_own']
        + summaries['repair_vs_random']
        + summaries['induction_success_minus_failed']
    ) / 3.0
    summaries = summaries.sort_values(
        ['selection_score', 'target_alignment_vs_own', 'layer_index'],
        ascending=[False, False, True],
    )
    selected = summaries.iloc[0].to_dict()
    frozen_stage_by_edge[edge_type] = str(selected['stage'])
    for row in summaries.to_dict(orient='records'):
        row.update({
            'edge_type': edge_type,
            'selected': bool(row['stage'] == selected['stage']),
            'discovery_families': int(current['family_id'].nunique()),
            'discovery_pairs': int(current['pair_id'].nunique()),
        })
        selection_rows.append(row)

assert frozen_stage_by_edge, 'No edge type had discovery support.'
selection_frame = pd.DataFrame(selection_rows)
selection_frame.to_csv(ANALYSIS_DIR / 'discovery_stage_curves.csv', index=False)
discovery_document = {
    'schema_version': 'outcomeflip-fracture-discovery-selection-1.0',
    'active_pair_sha256': sha256_file(ACTIVE_PAIR_MANIFEST_PATH),
    'frozen_stage_by_edge': frozen_stage_by_edge,
    'selection_metric': (
        'mean of repair_vs_own, repair_vs_random, '
        'and induction_success_minus_failed'
    ),
    'confirmation_results_used': False,
}
write_json_atomic(DISCOVERY_SELECTION_PATH, discovery_document)
print('Frozen stage by edge:', frozen_stage_by_edge)
display(selection_frame.round(5))


In [ ]:
%%cell_progress 15/17 Test reciprocal localization on confirmation families
cell_stage('Bootstrapping complete confirmation families at frozen stages')

CONFIRMATION_METRICS = (
    'repair_vs_own', 'repair_vs_random',
    'induction_success_minus_failed', 'target_alignment_vs_own',
    'repair_success_advantage', 'repair_minus_copy',
)

def family_cluster_interval(current, column, key):
    family_values = current.groupby('family_id')[column].mean().to_numpy(dtype=float)
    assert len(family_values) > 0
    generator = np.random.default_rng(
        int(stable_key(('bootstrap', key))[:16], 16) % (2**63 - 1)
    )
    indices = generator.integers(
        0, len(family_values), size=(BOOTSTRAP_REPLICATES, len(family_values))
    )
    draws = family_values[indices].mean(axis=1)
    return {
        'estimate': float(family_values.mean()),
        'ci_low': float(np.quantile(draws, ALPHA / 2)),
        'ci_high': float(np.quantile(draws, 1 - ALPHA / 2)),
    }

test_rows = []
decision_by_edge = {}
for edge_type, stage_name in frozen_stage_by_edge.items():
    current = confirmation_frame[
        (confirmation_frame['edge_type'] == edge_type)
        & (confirmation_frame['stage'] == stage_name)
    ].copy()
    n_families = int(current['family_id'].nunique())
    n_pairs = int(current['pair_id'].nunique())
    if current.empty:
        decision_by_edge[edge_type] = {
            'selected_stage': stage_name,
            'decision': 'confirmation_support_insufficient',
            'gates': {'adequate_confirmation_support': False},
            'n_families': 0, 'n_pairs': 0,
        }
        continue
    edge_results = {}
    for metric in CONFIRMATION_METRICS:
        result = family_cluster_interval(
            current, metric, ('confirmation', edge_type, stage_name, metric)
        )
        edge_results[metric] = result
        test_rows.append({
            'edge_type': edge_type, 'stage': stage_name,
            'metric': metric, **result,
            'n_families': n_families, 'n_pairs': n_pairs,
        })
    gates = {
        'adequate_confirmation_support': bool(
            n_families >= MINIMUM_CONFIRMATION_FAMILIES_PER_EDGE
        ),
        'successful_update_exceeds_failed_own': bool(
            edge_results['repair_vs_own']['ci_low'] > 0
        ),
        'successful_update_exceeds_random': bool(
            edge_results['repair_vs_random']['ci_low'] > 0
        ),
        'failed_update_impairs_successful_reference': bool(
            edge_results['induction_success_minus_failed']['ci_low'] > 0
        ),
        'repair_targets_recipient_not_reference_letter': bool(
            edge_results['target_alignment_vs_own']['ci_low'] > 0
        ),
    }
    if all(gates.values()):
        decision = 'reciprocal_fracture_stage_confirmed'
    elif gates['adequate_confirmation_support'] and (
        gates['successful_update_exceeds_failed_own']
        or gates['failed_update_impairs_successful_reference']
    ):
        decision = 'one_directional_stage_difference_only'
    elif not gates['adequate_confirmation_support']:
        decision = 'confirmation_support_insufficient'
    else:
        decision = 'matched_fracture_not_localized_at_selected_stage'
    decision_by_edge[edge_type] = {
        'selected_stage': stage_name, 'decision': decision,
        'gates': gates, 'n_families': n_families, 'n_pairs': n_pairs,
    }

test_frame = pd.DataFrame(test_rows, columns=[
    'edge_type', 'stage', 'metric', 'estimate', 'ci_low', 'ci_high',
    'n_families', 'n_pairs',
])
test_frame.to_csv(ANALYSIS_DIR / 'confirmation_tests.csv', index=False)

all_depth_rows = []
for (edge_type, stage_name), current in estimand_frame.groupby(
    ['edge_type', 'stage'], sort=True
):
    for metric in ('repair_vs_own', 'repair_vs_random',
                   'induction_success_minus_failed', 'target_alignment_vs_own'):
        result = family_cluster_interval(
            current, metric, ('all-depth', edge_type, stage_name, metric)
        )
        all_depth_rows.append({
            'edge_type': edge_type, 'stage': stage_name, 'metric': metric,
            **result, 'n_families': int(current['family_id'].nunique()),
            'n_pairs': int(current['pair_id'].nunique()),
        })
all_depth_frame = pd.DataFrame(all_depth_rows)
all_depth_frame.to_csv(ANALYSIS_DIR / 'all_support_depth_curves.csv', index=False)
write_json_atomic(ANALYSIS_DIR / 'fracture_decision.json', {
    'schema_version': 'outcomeflip-matched-fracture-decision-1.0',
    'decision_by_edge': decision_by_edge,
    'claim_boundary': (
        'The result localizes presentation-conditional update failures within '
        'Qwen-27B controlled torque outcome flips. It does not identify a '
        'universal stage for every model, task, or error type.'
    ),
})
display(test_frame.round(5))
print('Decisions:', json.dumps(decision_by_edge, indent=2))


In [ ]:
%%cell_progress 16/17 Audit matching, split, geometry, norms, and replay
cell_stage('Checking the causal scope and all preregistered controls')

pair_frame = pd.DataFrame(active_pair_manifest)
assert pair_frame['pair_id'].is_unique
assert set(pair_frame['edge_type']).issubset(set(EDGE_TYPES))
assert all(
    row['reference_edited_label'] != row['target_edited_label']
    for row in active_pair_manifest
)
assert all(
    int(row['reference_orientation'] != row['target_orientation'])
    + int(row['reference_mapping'] != row['target_mapping']) == 1
    for row in active_pair_manifest
)
discovery_families = set(
    pair_frame[pair_frame['split'] == 'discovery']['family_id']
)
confirmation_families = set(
    pair_frame[pair_frame['split'] == 'confirmation']['family_id']
)
assert not (discovery_families & confirmation_families)
assert len(record_frame) == len(active_pair_manifest) * len(STAGE_SITES) * 8
assert record_frame.groupby(['pair_id', 'stage']).size().eq(8).all()

norm_rows = []
for (pair_id, stage_name), current in estimand_frame.groupby(['pair_id', 'stage']):
    target_values = current.iloc[0]
    relative = abs(
        float(target_values['reference_update_norm'])
        - float(target_values['target_update_norm'])
    ) / max(float(target_values['target_update_norm']), 1e-12)
    norm_rows.append({
        'pair_id': pair_id, 'stage': stage_name,
        'target_reference_relative_norm_error': relative,
    })
norm_frame = pd.DataFrame(norm_rows)
assert float(norm_frame['target_reference_relative_norm_error'].max()) < NORM_RELATIVE_TOLERANCE
norm_frame.to_csv(ANALYSIS_DIR / 'norm_audit.csv', index=False)

audit_document = {
    'schema_version': 'outcomeflip-matched-fracture-audit-1.0',
    'candidate_pairs': len(pair_manifest),
    'active_pairs': len(active_pair_manifest),
    'active_families': int(pair_frame['family_id'].nunique()),
    'planned_minimum_active_pairs': MINIMUM_ACTIVE_PAIRS,
    'active_support_shortfall': ACTIVE_SUPPORT_SHORTFALL,
    'replay_concordance': REPLAY_CONCORDANCE,
    'pairs_by_split_edge': {
        f'{split}:{edge}': int(count)
        for (split, edge), count in pair_frame.groupby(
            ['split', 'edge_type']
        ).size().items()
    },
    'stale_error_pairs': int(pair_frame['stale_error'].sum()),
    'single_presentation_factor_difference': True,
    'reference_and_target_gold_letters_always_differ': True,
    'discovery_confirmation_family_overlap': 0,
    'cohort_selection_uses_intervention_outputs': False,
    'stage_selection_uses_confirmation_outputs': False,
    'maximum_full_state_reconstruction_error': reconstruction_error,
    'maximum_matched_norm_relative_error': float(
        norm_frame['target_reference_relative_norm_error'].max()
    ),
    'actual_accelerator': torch.cuda.get_device_name(0),
}
write_json_atomic(ANALYSIS_DIR / 'fracture_audit.json', audit_document)
display(audit_document)


In [ ]:
%%cell_progress 17/17 Save the fracture map, report, and completion manifest
cell_stage('Rendering depth curves and writing the paper-facing decision')
import matplotlib.pyplot as plt

stage_order = [site['stage'] for site in STAGE_SITES]
stage_labels = {
    'factor_update': 'L48\nfactor',
    'physical_handoff': 'L49\nphysical',
    'response_code': 'L55\ncode',
    'late_route': 'L57\nroute',
    'readout': 'L63\nreadout',
}
metrics_to_plot = (
    ('repair_vs_own', 'Successful minus failed update'),
    ('repair_vs_random', 'Successful minus random update'),
    ('induction_success_minus_failed', 'Success minus failed induction'),
    ('target_alignment_vs_own', 'Recipient-letter alignment'),
)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
colors = {'mapping': '#2563EB', 'orientation': '#D97706'}
for axis, (metric, title_text) in zip(axes.ravel(), metrics_to_plot):
    for edge_type in EDGE_TYPES:
        current = all_depth_frame[
            (all_depth_frame['edge_type'] == edge_type)
            & (all_depth_frame['metric'] == metric)
        ].set_index('stage').reindex(stage_order)
        if current['estimate'].isna().all(): continue
        x = np.arange(len(stage_order))
        y = current['estimate'].to_numpy(dtype=float)
        axis.errorbar(
            x, y,
            yerr=np.vstack([
                y - current['ci_low'].to_numpy(dtype=float),
                current['ci_high'].to_numpy(dtype=float) - y,
            ]),
            marker='o', capsize=4, label=edge_type.title(),
            color=colors[edge_type],
        )
        selected_stage = frozen_stage_by_edge.get(edge_type)
        if selected_stage in stage_order:
            axis.axvline(
                stage_order.index(selected_stage), color=colors[edge_type],
                linestyle=':', alpha=0.35,
            )
    axis.axhline(0, color='black', linewidth=0.8)
    axis.set_xticks(np.arange(len(stage_order)), [stage_labels[s] for s in stage_order])
    axis.set_title(title_text)
    axis.legend(frameon=False)
fig.suptitle('Matched outcome-flip fracture localization', fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.96))
figure_path = FIGURE_DIR / 'matched_fracture_depth_map.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
pdf_path = figure_path.with_suffix('.pdf')
try:
    fig.savefig(pdf_path, bbox_inches='tight')
    PDF_SAVED = True
except (ImportError, AttributeError) as error:
    PDF_SAVED = False
    print(f'PDF export skipped: {error}')
plt.show()

def interval(edge_type, metric):
    rows = test_frame[
        (test_frame['edge_type'] == edge_type)
        & (test_frame['metric'] == metric)
    ]
    if rows.empty: return 'not estimable'
    row = rows.iloc[0]
    return f"{row['estimate']:.5f} [{row['ci_low']:.5f}, {row['ci_high']:.5f}]"

lines_out = [
    '# Qwen-27B Matched Outcome-Flip Fracture Localization', '',
    '## Design', '',
    f'- Behavior-defined candidate pairs: {len(pair_manifest)}',
    f'- Replayed active pairs: {len(active_pair_manifest)}',
    f'- Active physical families: {pair_frame["family_id"].nunique()}',
    f'- Replay concordance: {REPLAY_CONCORDANCE:.3f}',
    '- Every pair differs in exactly one presentation factor.',
    '- Reference and target require different output letters.',
    '- Discovery and confirmation share no physical family.', '',
]
for edge_type in EDGE_TYPES:
    if edge_type not in decision_by_edge: continue
    result = decision_by_edge[edge_type]
    lines_out.extend([
        f'## {edge_type.title()} edge', '',
        f"- Frozen stage: {result['selected_stage']}",
        f"- Decision: {result['decision']}",
        f"- Confirmation families / pairs: {result['n_families']} / {result['n_pairs']}",
        f"- Repair versus failed own update: {interval(edge_type, 'repair_vs_own')}",
        f"- Repair versus random: {interval(edge_type, 'repair_vs_random')}",
        f"- Reciprocal induction: {interval(edge_type, 'induction_success_minus_failed')}",
        f"- Recipient-letter alignment: {interval(edge_type, 'target_alignment_vs_own')}",
        f"- Gates: {json.dumps(result['gates'], sort_keys=True)}", '',
    ])
lines_out.extend([
    '## Interpretation boundary', '',
    (
        'This experiment tests where presentation-conditional update failures '
        'become causally distinguishable within controlled Qwen-27B torque '
        'outcome flips. A negative result means the five frozen token-layer '
        'sites do not isolate the fracture; it does not mean physical '
        'information is absent everywhere in the network.'
    ), '',
])
report_path = ANALYSIS_DIR / 'MATCHED_FRACTURE_REPORT.md'
report_path.write_text('\n'.join(lines_out), encoding='utf-8')
run_complete = {
    'schema_version': 'outcomeflip-matched-fracture-complete-1.0',
    'status': 'complete',
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'active_pairs': len(active_pair_manifest),
    'active_families': int(pair_frame['family_id'].nunique()),
    'decision_by_edge': decision_by_edge,
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'active_pair_sha256': sha256_file(ACTIVE_PAIR_MANIFEST_PATH),
    'discovery_selection_sha256': sha256_file(DISCOVERY_SELECTION_PATH),
    'figure': str(figure_path),
    'pdf_figure': str(pdf_path) if PDF_SAVED else None,
    'report': str(report_path),
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
}
write_json_atomic(OUTPUT_ROOT / 'RUN_COMPLETE.json', run_complete)
print(report_path.read_text(encoding='utf-8'))
print('RUN_COMPLETE:', OUTPUT_ROOT / 'RUN_COMPLETE.json')
